```quote
Every operation that creates a new tensor in forward pass needs a backward rule so the gradient can flow through it
```

1. The Tensor class. 

The mathematical difference between the scaler Value is that gradients are no longer numbers, they're arrays of same shape as the tensor. 
When tensor of different shapes interact, gradients must be summed back down to original shape. 

For C = A + B where B is broadcast (eg bias vector added to a batch). 
A shape = (N, D)
B shape = (D, )

$$\frac{\partial L}{\partial B_j} = \sum_{i=1}^{N} \frac{\partial L}{\partial C_{ij}}$$

i.e I need to sum the incoming gradient over the broadcasted axes. This is only the new idea here, the chain rule itself is identical to what I already implemented in micrograd. 

In [ ]:
import numpy as np
class Tensor:
    def __init__(self, data, _children=(), _op=''):
        self.data = np.array(data, dtype=np.float64)
        self.grad = np.zeros_like(self.data)
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
    
    @property
    def shape(self):
        return self.data.shape
    
    def __repr__(self):
        return f"Tensor(data={self.data})"

    def _unbroadcast(self, grad, shape):
        # grad shape (3, ), target shape (). 
        # sums across the rows
        
        while grad.ndim > len(shape):
            grad = grad.sum(axis=0)
        
        # checks for 1 dimensions in target shape and sum across that axis.
        # if target shape is (1, 3) sums over that axis and keeps the dimension as (1, 3) 
        for i, dim in enumerate(shape):
            if dim == 1:
                grad = grad.sum(axis=i, keepdims=True)
        return grad

    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += self._unbroadcast(out.grad, self.data.shape)
            other.grad += self._unbroadcast(out.grad, other.data.shape)
        out._backward = _backward
        
        return out
    
    def __radd__(self, other):
        return self + other
    
    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += self._unbroadcast(out.grad * other.data, self.data.shape)
            other.grad += self._unbroadcast(out.grad * self.data, other.data.shape)
        
        out._backward = _backward
        return out
    
    def __rmul__(self, other):
        return self * other 
    
    def __matmul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data @ other.data, (self, other), '@')

        def _backward():
            self.grad += self._unbroadcast(out.grad @ np.swapaxes(other.data, -1, -2), self.data.shape)
            other.grad += self._unbroadcast(np.swapaxes(self.data, -1, -2) @ out.grad, other.data.shape)
        
        out._backward = _backward
        return out
    
    def sum(self, axis=None, keepdims=False):
        out = Tensor(self.data.sum(axis=axis, keepdims=keepdims), (self, ), 'sum')

        def _backward():
            grad = out.grad
            if axis is not None and not keepdims:
                # restoring collapsed axis shape
                grad = np.expand_dims(grad, axis)
            
            # the partial derivative of out wrt to any input element is 1. 
            self.grad += np.ones_like(self.data) * grad
        
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        self.grad = np.ones_like(self.data)
        for node in reversed(topo):
            node._backward()

        
        



In [ ]:
a = Tensor([1.0, 2.0, 3.0])
b = Tensor(2.0)
c = b + a
c

Tensor(data=[3. 4. 5.])

In [ ]:
print(f"a shape: {a.shape}, b shape: {b.shape}")

a shape: (3,), b shape: ()


In [ ]:
c.grad = np.array([1.0, 1.0, 1.0])
c._backward()
print(a.grad, b.grad)

[1. 1. 1.] 3.0


Since b was added to all 3 elements of a, it's gradient is the sum of all incoming gradients (1.0 + 1.0 + 1.0) = 3.0. 


In [5]:
d = Tensor([
    [1.0, 2.0],
    [3.0, 4.0]
])

e = Tensor([10.0, 20.0])
f = d + e
print("d shape", d.shape)
print("e shape", e.shape)
print("f shape", f.shape)
f

d shape (2, 2)
e shape (2,)
f shape (2, 2)


Tensor(data=[[11. 22.]
 [13. 24.]])

In [6]:
f.grad = np.array([
    [1.0, 2.0],
    [3.0, 4.0]
])

f._backward()

print("d grad: ", d.grad)
print("e grad: ", e.grad)

d grad:  [[1. 2.]
 [3. 4.]]
e grad:  [4. 6.]


here, 10.0 was added to both 1.0 and 3.0 so it must have gradient of both 1.0 and 3.0. (1.0 + 3.0 = 4.0)



In [ ]:
g = Tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0]
])

h = Tensor([
    [10.0, 20.0, 30.0]
])

i = g + h
print("g shape: ", g.shape)
print("h shape: ", h.shape)
print("i shape: ", i.shape)

g shape:  (2, 3)
h shape:  (1, 3)
i shape:  (2, 3)


numpy did this while adding. 

$$i = \begin{bmatrix} 1.0 & 2.0 & 3.0 \\ 4.0 & 5.0 & 6.0 \end{bmatrix} + \begin{bmatrix} 10.0 & 20.0 & 30.0 \\ 10.0 & 20.0 & 30.0 \end{bmatrix} = \begin{bmatrix} 11.0 & 22.0 & 33.0 \\ 14.0 & 25.0 & 36.0 \end{bmatrix}$$

In [ ]:
# suppose the incoming grad for i is. 
i.grad = np.array([
    [1.0, 1.0, 1.0],
    [2.0, 2.0, 2.0],
])

i._backward()

print("g grad: ", g.grad)
print("i grad: ", h.grad)

g grad:  [[1. 1. 1.]
 [2. 2. 2.]]
i grad:  [[3. 3. 3.]]


In [ ]:
# a complex multi-dimensional tensor shape
A = Tensor(np.ones((2, 4, 3)))

B_data = np.array([
    [[10.0, 20.0, 30.0]],
    [[40.0, 50.0, 60.0]]
])

B = Tensor(B_data)

C = A + B

print("A shape: ", A.shape)
print("B shape: ", B.shape)
print("C shape: ", C.shape)

A shape:  (2, 4, 3)
B shape:  (2, 1, 3)
C shape:  (2, 4, 3)


In [ ]:
A

Tensor(data=[[[1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]]

 [[1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]]])

In [ ]:
C

Tensor(data=[[[11. 21. 31.]
  [11. 21. 31.]
  [11. 21. 31.]
  [11. 21. 31.]]

 [[41. 51. 61.]
  [41. 51. 61.]
  [41. 51. 61.]
  [41. 51. 61.]]])

In [12]:
# suppose upstream gradient is. 
C.grad = np.ones((2, 4, 3))
C._backward()

print("A grad: ", A.grad)
print("B grad: ", B.grad)

A grad:  [[[1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]]

 [[1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]]]
B grad:  [[[4. 4. 4.]]

 [[4. 4. 4.]]]


2. Adding element-wise multiply, matrix multiply and backward() driver. 

- element-wise multiplication. 
suppose C = A*B. 

then $$\frac{\partial L}{\partial A} = \frac{\partial L}{\partial C} ⊙B $$ (then unboardcast)

- matrix multiply. 
Y = X @ W where X (N, D) and W (D, M) Y(N, M)

given upstream gradient $$\frac{\partial L}{\partial W} = X^{T} \frac{\partial L}{\partial Y}$$

$$\frac{\partial L}{\partial X} = \frac{\partial L}{\partial Y} W^{T} $$

here shapes must work out together. 

dX must be (N, D). 
so W must be transposed. same goes for dW

# 1. Element-wise matrix multiplication and gradient flow

In [ ]:
A = Tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0]
])

B  = Tensor([2.0, 3.0, 4.0])

C = A*B # element wise multiply. 

B is expanded to (1, 3) and then it is broadcasted to (2, 3) to match size of A. C has now shape (2, 3). 

suppose C.grad is all-ones matrix. 

In [14]:
C.grad = np.array([
    [1.0, 1.0, 1.0],
    [1.0, 1.0, 1.0]
])

C._backward()

print("A.grad: ", A.grad)
print("B.grad: ", B.grad)

A.grad:  [[2. 3. 4.]
 [2. 3. 4.]]
B.grad:  [5. 7. 9.]


When you look at the _backward driver code. 
here out.grad is C.grad. self is A and other is B. 
so dA = out.grad * B.data = (2, 3) * (3, ) = (2, 3) that matches with shape of A so no unbroadcasting is needed. 

Whereas. 
db = out.grad * A.data = (2, 3) * (2, 3) = (2, 3) but it is need to be (3, ). 
so the grad along dimension 2 must be accumulate. so unbroadcasting is needed. 

here look at the unbroadcast code now. 
grad = (2, 3), shape = (3, )
here first while condition match, grad has more dimensions. 
    so along axis=0, that is along dimension 2, values are added, that returns (3, ) has final output. 




# 2. Matrix multiplication & gradient flow

In [ ]:
A = Tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0]
]) # (2, 3)

B  = Tensor([
    [2.0, 3.0],
    [4.0, 5.0],
    [6.0, 7.0]
]) # (3, 2)

# both matrices' inner dim matches so they can be multiplied. 

C = A @ B # has shape (2, 2). 

print("C.shape: ", C.shape)
print("C.data:\n", C.data)


C.shape:  (2, 2)
C.data:
 [[28. 34.]
 [64. 79.]]


In [ ]:
# lets assume C.grad to be
C.grad = np.array([
    [1.0, 1.0],
    [1.0, 1.0]
])
C._backward()

print("A.grad: \n", A.grad)
print("B.grad: \n", B.grad)

A.grad: 
 [[ 5.  9. 13.]
 [ 5.  9. 13.]]
B.grad: 
 [[5. 5.]
 [7. 7.]
 [9. 9.]]


If you do that on paper, you will get the same grad as above.

Note: mat mul works for 3D/4D tensors, because @ automatically handles batched matrix multiplication broadcasting across higher dimensions. 

but backward pass fails. 
Below code document this limitation. Take a look!

In [ ]:
X_3d  = Tensor(np.ones((2, 3, 4))) # 2 batch of (3, 4) matrix
W_3d = Tensor(np.ones((2, 4, 5))) # 2 batch of (4, 5) matrix

Y_3d = X_3d @ W_3d

print("Y_3d.shape: ", Y_3d.shape)
print("Y_3d.data: \n", Y_3d.data)

try:
    loss = Y_3d.sum()
    print("loss: ", loss)
    loss.backward()
    print("X_3d.grad shape: ", X_3d.grad.shape)
except Exception as e:
    print("Backward pass fails as expected")
    print("error: ", str(e))

Y_3d.shape:  (2, 3, 5)
Y_3d.data: 
 [[[4. 4. 4. 4. 4.]
  [4. 4. 4. 4. 4.]
  [4. 4. 4. 4. 4.]]

 [[4. 4. 4. 4. 4.]
  [4. 4. 4. 4. 4.]
  [4. 4. 4. 4. 4.]]]
loss:  Tensor(data=120.0)
X_3d.grad shape:  (2, 3, 4)


The reason it fails, while dX_3d = loss.grad(2, 3, 5) @ W_3d(2, 4, 5)(T) 
W_3d is transpose along all axis (2, 4, 5) -> (5, 4, 2) and thus we get "size 4 is different from 5" value error. 

so instead of swapping the matrix axes to (2, 5, 4) using np.swapaxes(W, -1, -2). 

also what if the sample size or batch size is broadcasted while computing loss? 
suppose X_3d has (2, 3, 4) and W_3d has (1, 4, 5) then, 
numpy does obtain (2, 3, 5) result. But backward pass fails. 
dW_3d grad will have (assume above fix is applied) shape (2, 4, 5) but W_3d must have target shape of (1, 4, 5) so need to perform unbroadcast here. 

So this is fixed by the intended code above in the __matmul__ operation. 
now the above code is working as expected. 

# 3. nn.Module, nn.Parameter, nn.Linear

Parameter is just a Tensor with a special marker so Module can auto-discover it. 

Module.parameters() walks self.__dict__, collecting any Parameters and recursing into sub-Modules. 

In [ ]:
class Parameter(Tensor):
    def __init__(self, data):
        super().__init__(data)

class Module:
    # returns a list of individual Parameter tensor objects, not the total count of scalar numerical weights.
    def parameters(self):
        params = []
        for attr in vars(self).values():
            if isinstance(attr, Parameter):
                params.append(attr)
            elif isinstance(attr, Module):
                params.extend(attr.parameters())
            elif isinstance(attr, (list, tuple)):
                for item in attr:
                    if isinstance(item, Module):
                        params.extend(item.parameters())
        return params
    
    def zero_grad(self):
        for p in self.parameters():
            p.grad = np.zeros_like(p.data)
    
    def __call__(self, *args, **kwargs):
        return self.forward(*args, **kwargs)
    
    def forward(self, *args, **kwargs):
        raise NotImplementedError

class Linear(Module):
    def __init__(self, din, dout, bias=True):
        bound = 1 / np.sqrt(din)
        self.weight = Parameter(np.random.uniform(-bound, bound, (din, dout)))
        self.bias = Parameter(np.random.uniform(-bound, bound, (dout, ))) if bias else None
    
    def forward(self, x):
        out = x @ self.weight
        if self.bias is not None:
            out = out + self.bias
        return out
    
    def __repr__(self):
        in_f, out_f = self.weight.data.shape
        return f"Linear(in_features={in_f}, out_features={out_f})"


In [27]:
np.random.seed(0)

layer = Linear(3, 4)
print(layer)

Linear(in_features=3, out_features=4)


In [37]:
vars(layer).values()

dict_values([Tensor(data=[[ 0.05636498  0.24847928  0.11866093  0.05182664]
 [-0.08815584  0.16846401 -0.07206808  0.45238049]
 [ 0.53539164 -0.13459014  0.33685506  0.03336498]]), Tensor(data=[ 0.07857109  0.49143667 -0.49532489 -0.47674202])])

In [31]:
print("layer.weight: \n", layer.weight.data)
print("layer.bias: \n", layer.bias.data)
print("num params: ", len(layer.parameters()))
print("total individual weights: ", sum(p.data.size for p in layer.parameters()))

layer.weight: 
 [[ 0.05636498  0.24847928  0.11866093  0.05182664]
 [-0.08815584  0.16846401 -0.07206808  0.45238049]
 [ 0.53539164 -0.13459014  0.33685506  0.03336498]]
layer.bias: 
 [ 0.07857109  0.49143667 -0.49532489 -0.47674202]
num params:  2
total individual weights:  16


In [29]:
x = Tensor(np.random.randn(2, 5, 3))
y = layer(x)

print("output shape: ", y.shape)

output shape:  (2, 5, 4)


In [30]:
loss = y.sum()
print("loss: ", loss)
loss.backward()

print("weight.grad shape: ", layer.weight.grad.shape)
print("bias.grad shape: ", layer.bias.grad.shape)

loss:  Tensor(data=-2.7348525072291485)
weight.grad shape:  (3, 4)
bias.grad shape:  (4,)
